<a href="https://colab.research.google.com/github/mjmj002/Sparta/blob/main/Python/Project/01_%EB%8D%B0%EC%9D%B4%ED%84%B0%EC%88%98%EC%A7%91_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1~2 | 데이터 수집 & 가공
> **Team Sparta | 주식/코인 모니터링 웹앱 프로젝트**

---

## 이번 노트북에서 만들 것
1. `yfinance` / `pyupbit` API로 실제 데이터 받아오기
2. 종목 정보 조회 함수 만들기
3. 기간별 가격 히스토리 함수 만들기
4. 등락률 / 이동평균 계산 함수 만들기

> **빈칸(`___`)을 채우면서 진행하세요. 힌트는 주석을 확인하세요!**

## 환경 준비

In [ ]:
# 패키지 설치 (최초 1회만 실행)
!pip install yfinance streamlit plotly pandas pyupbit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.9 MB/s eta 0:00:00


In [ ]:
# 필요한 라이브러리를 import 하세요
import yfinance as yf          # 힌트: y로 시작하는 주식 라이브러리
import pandas as pd      # 힌트: 보통 pd로 줄여 씀
import numpy as np          # 힌트: 수치 계산 라이브러리
from datetime import datetime, timedelta

print('라이브러리 import 완료!')

라이브러리 import 완료!


---
##  yfinance API 탐색
본격적으로 함수를 만들기 전에, 데이터가 어떻게 생겼는지 먼저 확인합니다.

In [ ]:
# 탐색용 코드 — 수정하지 말고 그대로 실행해서 구조를 확인하세요

ticker = yf.Ticker('AAPL')

# info: 종목의 다양한 정보가 담긴 딕셔너리
info = ticker.info
print('=== 사용 가능한 키 목록 (일부) ===')
keys_to_show = ['currentPrice', 'previousClose', 'shortName', 'currency', 'marketCap']
for key in keys_to_show:
    print(f'  {key}: {info.get(key)}')

=== 사용 가능한 키 목록 (일부) ===
  currentPrice: 251.64
  previousClose: 251.49
  shortName: Apple Inc.
  currency: USD
  marketCap: 3698586025984


In [ ]:
#  히스토리 데이터 구조 확인
hist = ticker.history(period='1mo')  # 최근 1달

print('=== DataFrame 컬럼 ===')
print(hist.columns.tolist()) # 리스트 형태로
print()
print('=== 최근 5일 데이터 ===')
print(hist.tail())
print()
print(f'행 수: {len(hist)}, 컬럼 수: {len(hist.columns)}')

=== DataFrame 컬럼 ===
['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']

=== 최근 5일 데이터 ===
                                 Open        High         Low       Close  \
Date                                                                        
2026-03-18 00:00:00-04:00  252.630005  254.940002  249.000000  249.940002   
2026-03-19 00:00:00-04:00  249.399994  251.830002  247.300003  248.960007   
2026-03-20 00:00:00-04:00  247.979996  249.199997  246.000000  247.990005   
2026-03-23 00:00:00-04:00  253.970001  254.600006  250.279999  251.490005   
2026-03-24 00:00:00-04:00  250.490005  254.824997  249.550003  251.639999   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-03-18 00:00:00-04:00  35757900        0.0           0.0  
2026-03-19 00:00:00-04:00  34864100        0.0           0.0  
2026-03-20 00:00:00-04:00  88331100        0.0           0.0  
2026-03-23 00:00:00-04:00  405461

In [ ]:
# period 옵션 종류 확인
# period 값: '1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y'

for period in ['1wk', '1mo', '3mo', '1y']:
    df = ticker.history(period=period)
    print(f'period={period!r:8s} → {len(df):3d}행,  기간: {df.index[0].date()} ~ {df.index[-1].date()}')

period='1wk'    →   5행,  기간: 2026-03-18 ~ 2026-03-24
period='1mo'    →  20행,  기간: 2026-02-25 ~ 2026-03-24
period='3mo'    →  60행,  기간: 2025-12-26 ~ 2026-03-24
period='1y'     → 251행,  기간: 2025-03-25 ~ 2026-03-24


---
## 함수 1 | `get_stock_info()` — 종목 현재 정보 조회

### 요구사항
- 입력: 티커 심볼 (예: `'AAPL'`, `'005930.KS'`)
- 출력: 아래 키를 포함한 딕셔너리

```python
{
    'name': '애플',          # 종목명
    'current_price': 182.5,  # 현재가
    'prev_close': 180.0,     # 전일 종가
    'currency': 'USD',       # 통화
}
```
- 잘못된 티커나 오류 시 `None` 반환 + 오류 메시지 출력

In [ ]:
def get_stock_info(ticker_symbol):
    """
    주식 종목의 현재 정보를 조회합니다.

    Args:
        ticker_symbol (str): 티커 심볼 (예: 'AAPL', '005930.KS')

    Returns:
        dict: 종목 정보 딕셔너리 또는 오류 시 None
    """
    try:
        # 1단계: yf.Ticker 객체 생성
        # 힌트: yf.Ticker(___) 형태로 작성
        ticker = yf.Ticker(ticker_symbol)

        # 2단계: ticker.info 가져오기
        # 힌트: 위 탐색 코드에서 봤던 방법 그대로
        info = ticker.info

        # 3단계: 필요한 값 추출 (없으면 기본값 반환)
        # 힌트: info.get('키이름', 기본값) 사용
        result = {
            'name':          info.get('shorkName', ticker_symbol),   # 힌트: 'shortName'
            'current_price': info.get('currentPrice', 0),               # 힌트: 'currentPrice'
            'prev_close':    info.get('previousClose', 0),               # 힌트: 'previousClose'
            'currency':      info.get('currency', 'USD'),           # 힌트: 'currency'
        }

        # 4단계: current_price가 0이면 데이터가 없는 것 → None 반환
        if result['current_price'] == 0:
            print(f'[경고] {ticker_symbol}: 가격 데이터를 찾을 수 없습니다.')
            return None

        return result

    except Exception as e:
        # 힌트: f-string으로 ticker_symbol과 e를 포함한 오류 메시지 출력
        print(f'[오류] {ticker_symbol}: {e}')
        return None

info = get_stock_info('AAPL')
print(info)

{'name': 'AAPL', 'current_price': 251.64, 'prev_close': 251.49, 'currency': 'USD'}


In [ ]:
# 함수 테스트
print('=== 정상 케이스 ===')
result = get_stock_info('AAPL')
print(result)

print()
print('=== 한국 주식 ===')
result_kr = get_stock_info('005930.KS')  # 삼성전자
print(result_kr)

print()
print('=== 오류 케이스 (잘못된 티커) ===')
result_err = get_stock_info('INVALID_TICKER_XYZ')
print(result_err)  # None이 출력되어야 함

=== 정상 케이스 ===
{'name': 'AAPL', 'current_price': 251.64, 'prev_close': 251.49, 'currency': 'USD'}

=== 한국 주식 ===
{'name': '005930.KS', 'current_price': 189000.0, 'prev_close': 189700.0, 'currency': 'KRW'}

=== 오류 케이스 (잘못된 티커) ===


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALID_TICKER_XYZ"}}}


[경고] INVALID_TICKER_XYZ: 가격 데이터를 찾을 수 없습니다.
None


---
## 함수 2 | `get_stock_history()` — 기간별 가격 히스토리

### 요구사항
- 입력: 티커 심볼, 기간 문자열
- 출력: `Open`, `High`, `Low`, `Close`, `Volume` 컬럼이 있는 DataFrame
- 기간 매핑:

```
'1주' → '1wk'
'1달' → '1mo'
'3달' → '3mo'
'1년' → '1y'
```

In [ ]:
def get_stock_history(ticker_symbol, period='1달'):
    """
    주식 종목의 기간별 가격 히스토리를 조회합니다.

    Args:
        ticker_symbol (str): 티커 심볼
        period (str): 조회 기간 ('1주', '1달', '3달', '1년')

    Returns:
        pd.DataFrame: OHLCV 데이터 또는 오류 시 None
    """
    # 기간 매핑 딕셔너리
    # 힌트: 키는 한글('1주', '1달', '3달', '1년'), 값은 yfinance period 문자열
    period_map = {
        '1주': '1wk',
        '1달': '1mo',
        '3달': '3mo',
        '1년': '1y',
    }

    # period_map에서 yfinance 기간값 꺼내기 (없으면 '1mo' 기본값)
    # 힌트: period_map.get(period, '1mo')
    yf_period = period_map.get(period, '1mo')

    try:
        ticker = yf.Ticker(ticker_symbol)

        # history() 메서드로 데이터 조회
        # 힌트: ticker.history(period=___)
        df = ticker.history(period=yf_period)

        # 데이터가 비어있는지 확인
        # 힌트: df.empty 속성 사용
        if df.empty:
            print(f'[경고] {ticker_symbol}: 히스토리 데이터가 없습니다.')
            return None

        # 필요한 컬럼만 선택해서 반환
        # 힌트: df[['Open', 'High', 'Low', 'Close', 'Volume']]
        return df[['Open', 'High', 'Low', 'Close', 'Volume']]

    except Exception as e:
        print(f'[오류] {ticker_symbol} 히스토리 조회 실패: {e}')
        return None

In [ ]:
# 함수 테스트
df = get_stock_history('AAPL', '1달')
print(f'행 수: {len(df)}')
print(df.tail(3))

print()
# 여러 기간 테스트
for period in ['1주', '1달', '3달', '1년']:
    df = get_stock_history('TSLA', period)
    print(f'기간={period} → {len(df)}행')

행 수: 20
                                 Open        High         Low       Close  \
Date                                                                        
2026-03-20 00:00:00-04:00  247.979996  249.199997  246.000000  247.990005   
2026-03-23 00:00:00-04:00  253.970001  254.600006  250.279999  251.490005   
2026-03-24 00:00:00-04:00  250.490005  254.824997  249.550003  251.639999   

                             Volume  
Date                                 
2026-03-20 00:00:00-04:00  88331100  
2026-03-23 00:00:00-04:00  40546100  
2026-03-24 00:00:00-04:00  27882961  

기간=1주 → 5행
기간=1달 → 20행
기간=3달 → 60행
기간=1년 → 251행


---
## 함수 3 | `calculate_change()` — 등락가 & 등락률 계산

### 요구사항
- 입력: 현재가, 전일 종가
- 출력: `(등락가, 등락률)` 튜플
- 등락률 = `(현재가 - 전일종가) / 전일종가 × 100`
- 소수점 2자리 반올림

In [ ]:
def calculate_change(current_price, prev_close):
    """
    전일 대비 등락가와 등락률을 계산합니다.

    Args:
        current_price (float): 현재가
        prev_close (float): 전일 종가

    Returns:
        tuple: (등락가, 등락률) — 소수점 2자리
    """
    # 0으로 나누기 방지
    if prev_close == 0:
        return 0, 0

    # 등락가 계산: 현재가 - 전일종가
    # 힌트: 뺄셈 연산
    change = current_price - prev_close

    # 등락률 계산: (등락가 / 전일종가) × 100
    # 힌트: change / prev_close * 100
    change_pct = (change / prev_close) * 100

    # round()로 소수점 2자리 반올림
    return round(change, 2), round(change_pct, 2)

    # 테스트
diff, pct = calculate_change(20000,15000)
print(f"등락가 : {diff}, 등락률 : {pct}%")

등락가 : 5000, 등락률 : 33.33%


In [ ]:
# 함수 테스트
change, change_pct = calculate_change(185.0, 180.0)
print(f'등락가: {change}, 등락률: {change_pct}%')  # 5.0, 2.78%

change2, change_pct2 = calculate_change(175.0, 180.0)
print(f'등락가: {change2}, 등락률: {change_pct2}%')  # -5.0, -2.78%

# 실제 데이터로 테스트
info = get_stock_info('AAPL')
if info:
    c, cp = calculate_change(info['current_price'], info['prev_close'])
    print(f'AAPL 실제 등락가: {c}, 등락률: {cp}%')

등락가: 5.0, 등락률: 2.78%
등락가: -5.0, 등락률: -2.78%
AAPL 실제 등락가: 0.15, 등락률: 0.06%


---
## 함수 4 | `calculate_moving_average()` — 이동평균 계산 (선택)

### 요구사항
- 입력: 가격 히스토리 DataFrame, 이동평균 기간 리스트
- 출력: 이동평균 컬럼이 추가된 DataFrame
- 예: `periods=[5, 20]` → `MA5`, `MA20` 컬럼 추가

In [ ]:
def calculate_moving_average(df, periods=[5, 20]):
    """
    이동평균선을 계산해서 DataFrame에 컬럼으로 추가합니다.

    Args:
        df (pd.DataFrame): OHLCV 데이터
        periods (list): 이동평균 기간 리스트 (예: [5, 20, 60])

    Returns:
        pd.DataFrame: 이동평균 컬럼이 추가된 DataFrame
    """
    # DataFrame 복사 (원본 수정 방지)
    # 힌트: df.copy()
    result = df.copy()

    for period in periods:
        # 컬럼 이름: 'MA5', 'MA20' 등 f-string으로 만들기
        col_name = f'MA{period}'

        # pandas rolling().mean()으로 이동평균 계산
        # 힌트: result['Close'].rolling(window=period).mean()
        result[col_name] = result['Close'].rolling(window=period).mean()

    return result

In [ ]:
# 함수 테스트
df = get_stock_history('AAPL', '3달')
df_with_ma = calculate_moving_average(df, periods=[5, 20])

print('컬럼 목록:', df_with_ma.columns.tolist())
print(df_with_ma[['Close', 'MA5', 'MA20']].tail(10))

컬럼 목록: ['Open', 'High', 'Low', 'Close', 'Volume', 'MA5', 'MA20']
                                Close         MA5        MA20
Date                                                         
2026-03-11 00:00:00-04:00  260.809998  259.853998  264.317000
2026-03-12 00:00:00-04:00  255.759995  258.947995  263.330000
2026-03-13 00:00:00-04:00  250.119995  257.479996  262.749499
2026-03-16 00:00:00-04:00  252.820007  256.067996  262.601499
2026-03-17 00:00:00-04:00  254.229996  254.747998  262.118999
2026-03-18 00:00:00-04:00  249.940002  252.573999  261.398499
2026-03-19 00:00:00-04:00  248.960007  251.214001  260.817500
2026-03-20 00:00:00-04:00  247.990005  250.788004  259.988000
2026-03-23 00:00:00-04:00  251.490005  250.522003  259.253501
2026-03-24 00:00:00-04:00  251.639999  250.004004  258.228500


---
## (선택) 코인 데이터 — pyupbit
주식 대신 코인으로 만들고 싶다면 아래 함수를 사용하세요.

In [ ]:
# pyupbit 탐색
try:
    import pyupbit

    # 현재가 조회
    price = pyupbit.get_current_price('KRW-BTC')
    print(f'비트코인 현재가: {price:,}원')

    # OHLCV 조회 (count: 최근 N일)
    df_coin = pyupbit.get_ohlcv('KRW-BTC', count=30)
    print(df_coin.tail(3))

except ImportError:
    print('pyupbit 미설치: !pip install pyupbit 실행 후 재시도')

In [ ]:
def get_coin_history(ticker='KRW-BTC', days=30):
    """
    코인 OHLCV 데이터를 조회합니다.

    Args:
        ticker (str): 업비트 티커 (예: 'KRW-BTC', 'KRW-ETH')
        days (int): 조회 일수

    Returns:
        pd.DataFrame: OHLCV 데이터
    """
    try:
        import pyupbit
        # 힌트: pyupbit.get_ohlcv(ticker, count=days)
        df = ___

        # 컬럼명을 yfinance와 동일하게 통일
        # pyupbit: open/high/low/close/volume → Open/High/Low/Close/Volume
        df.columns = [col.capitalize() for col in df.columns]
        return df
    except Exception as e:
        print(f'[오류] 코인 데이터 조회 실패: {e}')
        return None

---
## Step 1~2 마무리 체크

아래 항목을 모두 확인한 후 다음 노트북으로 이동하세요!

| 체크 | 항목 |
|:---:|------|
| ☐ | `get_stock_info('AAPL')` 실행 시 딕셔너리 반환 |
| ☐ | `get_stock_info('INVALID')` 실행 시 None 반환 |
| ☐ | `get_stock_history('AAPL', '1달')` 실행 시 DataFrame 반환 |
| ☐ | 4가지 기간 모두 정상 작동 |
| ☐ | `calculate_change(185, 180)` → `(5.0, 2.78)` |
| ☐ | `calculate_moving_average()` 실행 시 MA5, MA20 컬럼 생성 |

### 다음 단계
`02_시각화_실습.ipynb` 으로 이동해서 plotly 차트를 만들어보세요!